# PySole Quickstart Tutorial: Wurtenkees (WUK) Bedrock Reconstruction

This tutorial demonstrates how to use `PySole` for physically-informed subglacial bedrock elevation reconstruction, 3D Eikonal ray migration, Universal Kriging drift selection via the `DriftAnalyzer`, and basal shear stress ($	au_b$) evaluation using the **Wurtenkees (WUK)** benchmark dataset.

## 1. Import PySole & Inspect Version

In [ ]:
import pysole
print(f"PySole Package Version: {pysole.__version__}")

## 2. High-Level One-Liner Execution (`run_from_config`)

Run the complete Wurtenkees processing pipeline from a configuration file in a single line:

In [ ]:
# Execute full glaciological processing pipeline
bedrock_map = pysole.run_from_config("wuk/pysole_wuk.json")

# Inspect spatial grid metadata
print(f"Grid Shape: {bedrock_map.shape}")
print(f"Bounding Box: {bedrock_map.bounds}")

## 3. Programmatic API with `Solver` & Interactive `DriftAnalyzer`

In [ ]:
from pysole import Solver

# Instantiate Solver orchestrator
solver = Solver.from_config("wuk/pysole_wuk.json")

# Run Eikonal 3D ray migration
solver.migrate_eikonal()

# Run surface slope BSS variogram optimization
solver.optimize_bss()

# Recommend optimal Universal Kriging drift model via DriftAnalyzer.
# Returns a list of CandidateDriftResult objects ranked best-first (profile-wise LOPO
# cross-validation is used automatically when `survey_profile_column` is configured).
import pandas as pd

results = solver.recommend_drift_model(stage="post_migration")
best = results[0]
print(f"Recommended Drift Model: {best.name} -> terms = {best.terms}")

df_rankings = pd.DataFrame(
    [{"name": r.name, "terms": r.terms, "cv_rmse": r.cv_rmse, "aicc": r.aicc,
      "vif_max": r.vif_max, "penalized": r.is_penalized} for r in results]
)
df_rankings.head()

## 4. Final Bedrock Calculation & Raster Export

In [ ]:
# Calculate bedrock elevation grid & blend perimeter margins
solver.calculate_bedrock()
final_bedrock = solver.finalize_bedrock()

# Save predicted bedrock GeoTIFF
final_bedrock.save("wuk_bedrock_tutorial.tif")